# HMK / 01 — Hierarchical Markov regimes: daily → 1 hour → 15 minutes on MNQ

Protocol: [`HMK 001`](../../discussions/MNQ_Hierarchical_Markov/001_claude_hmk01_preregistration.md). Code: `src/equity_trading/hmk.py`. Ticks per contract (1 tick = 0.25 pt = $0.50).

- **Tiers:** 3-state Gaussian HMM on daily returns (CME day); 3-state HMMs on 1-hour and 15-minute returns whose transition matrix depends on the parent's one-step forecast: P(s_t = j | s_{t-1} = i, parent) = Σ_g w_t[g] A_g[i, j].
- **Variants:** `H3` (hierarchy, primary), `U` (no parent: the benchmark for G1), `RULE` (1 h only: parent = the video's 20-day ±5% state of the previous day).
- **Forecast for bar t** at the close of t−1: P(up), E[r]. **Trade:** always in sign(E[r]) from the 2nd bar of each CME day, flat at the day's close; cost 1.5 ticks per contract traded at market (3 per round trip), 0.8 realized reported.
- **Gates (1 h primary, 15 m secondary):** G1 log score beats `U`; G2 Brier skill vs the training up-share; G3 net > 0 and beats always-long; G4 net > 0 and BSS > 0 in ≥ 2 of 3 periods. One-sided p < 0.05, Newey-West on daily series.
- **Windows:** discovery = expanding fits from 2020-01 refitted 2021-01, 2022-01, 2023-01, evaluated 2021-01 – 2023-06. `HMK_CONFIRM=1`: confirmation 2023-07 – 2025-12 (only after the discovery readout). 2026 sealed.

Environment: `HMK_SYNTHETIC=1` (planted structure), with `HMK_SYNTH_NULL=1` (no structure); `HMK_SYNTH_DIR`; `HMK_CONFIRM=1`.

In [ ]:
import json, os, sys, tempfile, pickle, time
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("HMK_SYNTHETIC") == "1"
SYNTH_NULL = os.environ.get("HMK_SYNTH_NULL") == "1"
CONFIRM = os.environ.get("HMK_CONFIRM") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("HMK_SYNTH_DIR") or tempfile.mkdtemp(prefix="hmk_")
    os.environ["HMK_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import hmk as H
from equity_trading import micro_bars as MB

D = dt.date
if SYNTHETIC:
    CFG = H.HMKConfig(n_starts=3, refits_discovery=(D(2020, 7, 1), D(2021, 1, 1)), discovery=(D(2020, 7, 1), D(2021, 6, 30)),
                      periods_discovery=(("2020Q3", D(2020, 7, 1), D(2020, 9, 30)), ("2020Q4", D(2020, 10, 1), D(2020, 12, 31)),
                                         ("2021H1", D(2021, 1, 1), D(2021, 6, 30))))
    if not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
        H.write_synthetic(os.environ["EQUITY_TRADING_DIR"], CFG.load_start, CFG.discovery[1], effect=0.0 if SYNTH_NULL else 2.0)
else:
    CFG = H.HMKConfig()
REFITS, WINDOW, PERIODS = ((CFG.refits_confirm, CFG.confirm, CFG.periods_confirm) if CONFIRM
                           else (CFG.refits_discovery, CFG.discovery, CFG.periods_discovery))
OUT = Path("01c" if CONFIRM else "01"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 80)
print("engine", H.HMK_VERSION, "| synthetic " + ("null" if SYNTH_NULL else "planted") if SYNTHETIC else "| real data",
      "| window", WINDOW, "| refits", REFITS, "| round trip", CFG.round_trip_market, "ticks")

## 1. Bars

In [ ]:
m1 = MB.load_1m(CFG.root, CFG.load_start, WINDOW[1] + dt.timedelta(days=1))
BARS = H.session_bars(m1, CFG)
del m1
cov = []
for tier, b in BARS.items():
    y = pd.to_datetime(b.trade_date).dt.year
    cov.append(b.groupby(y).agg(rows=("obs", "size"), observed=("obs", "mean"), days=("trade_date", "nunique"))
               .assign(tier=tier).reset_index(names="year"))
cov = pd.concat(cov); cov.to_csv(OUT / "01_coverage.csv", index=False)
display(cov.pivot_table(index="year", columns="tier", values=["days", "observed"]).round(3))

## 2. Walk-forward fits (cached)

In [ ]:
CACHE = OUT / f"wf_{H.HMK_VERSION}.pkl"
if CACHE.exists():
    WF = pickle.loads(CACHE.read_bytes())
else:
    T0 = time.time()
    WF = H.walk_forward(BARS, CFG, REFITS, WINDOW[1], log=lambda m: print(m, f"{time.time() - T0:.0f}s", flush=True))
    CACHE.write_bytes(pickle.dumps(WF))
json.dump(WF["fits"], open(OUT / "01_fits.json", "w"), indent=1, default=str)
for r, f in WF["fits"].items():
    print(r, "| daily mu/sd bp", f["1d"]["mu_bp"], f["1d"]["sd_bp"], "| 1h_H3", f["1h_H3"]["mu_bp"], f["1h_H3"]["sd_bp"],
          "| 15m_H3", f["15m_H3"]["mu_bp"], f["15m_H3"]["sd_bp"])

## 3. The conditional matrices — P(hour state | previous hour state, daily state)

States are sorted by emission mean (bear < side < bull). The 'Bayesian' joint is P(hour = j, day = g | hour_{t-1} = i) = P(day = g) · A_g[i, j]; the table also gives P(hour bear | day bull) at the stationary hour distribution of each daily state.

In [ ]:
last = list(WF["fits"])[-1]
CM = H.conditional_matrices(WF["fits"][last], "1h_H3")
CM.to_csv(OUT / "01_conditional_1h.csv", index=False)
display(CM.pivot_table(index=["parent", "from"], columns="to", values="p")[["bear", "side", "bull"]].round(3))
A = np.asarray(WF["fits"][last]["1h_H3"]["A"])
stat = []
for g in range(A.shape[0]):
    w, v = np.linalg.eig(A[g].T); p = np.real(v[:, np.argmin(np.abs(w - 1))]); p = p / p.sum()
    stat.append(dict(zip(["bear", "side", "bull"], p.round(3)), day_state=["bear", "side", "bull"][g]))
display(pd.DataFrame(stat).set_index("day_state"))
CM15 = H.conditional_matrices(WF["fits"][last], "15m_H3"); CM15.to_csv(OUT / "01_conditional_15m.csv", index=False)

## 4. Gates

In [ ]:
PRED = {k: H.in_window(v, *WINDOW) for k, v in WF["pred"].items()}
ROWS, PER, DEC = [], [], {}
for key in ("1h_H3", "1h_U", "1h_RULE", "15m_H3", "15m_U"):
    tier = key.split("_")[0]
    bench = None if key.endswith("_U") else PRED[f"{tier}_U"]
    r = H.evaluate(PRED[key], bench, CFG, key); ROWS.append(r)
    per = []
    for name, a, b in PERIODS:
        pr = H.evaluate(H.in_window(PRED[key], a, b), None if bench is None else H.in_window(bench, a, b), CFG, key)
        pr["period"] = name; per.append(pr); PER.append(pr)
    DEC[key] = H.verdict(r, per, CFG)
RES = pd.DataFrame(ROWS).set_index("label"); RES.to_csv(OUT / "01_gates.csv")
PERDF = pd.DataFrame(PER); PERDF.to_csv(OUT / "01_periods.csv", index=False)
show = ["bars", "days", "hit", "up_share", "mean_p_up", "bss", "g2_p", "g1_logscore_per_day", "g1_p", "gross_ticks_day",
        "units_day", "net_ticks_day", "g3_p", "net_realized_day", "long_net_day", "vs_long_ticks_day", "vs_long_p",
        "mean_abs_move_ticks", "breakeven_hit_if_every_bar_flips"]
display(RES[show].T.round(4))
display(PERDF.pivot_table(index="period", columns="label", values=["net_ticks_day", "bss", "hit"]).round(4))
print(json.dumps(DEC, indent=1))

## 5. Is P > 50% enough? Calibration and the law of large numbers

A forecast only pays if, after costs, (2p − 1) × E|move| exceeds the cost of the trades it causes. `breakeven_hit_if_every_bar_flips` above is the hit rate needed if the position flips every bar.

In [ ]:
CAL = pd.concat([H.calibration(PRED[k]).assign(model=k) for k in ("1h_H3", "1h_U", "15m_H3")])
CAL.to_csv(OUT / "01_calibration.csv", index=False); display(CAL.round(4))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for k in ("1h_H3", "1h_U", "15m_H3"):
    c = CAL[CAL.model == k]; ax[0].plot(c.p_up, c.realized_up, "o-", label=k)
ax[0].plot([0.4, 0.6], [0.4, 0.6], "k--", lw=0.8); ax[0].set_xlabel("forecast P(up)"); ax[0].set_ylabel("realized up share")
ax[0].set_title("Calibration (out of sample)"); ax[0].legend()
for k in ("1h_H3", "1h_RULE", "15m_H3"):
    p = H.daily_pnl(PRED[k], CFG); ax[1].plot(pd.to_datetime(p.index), p.net_market.cumsum(), label=f"{k} net")
p = H.daily_pnl(PRED["1h_H3"], CFG); ax[1].plot(pd.to_datetime(p.index), p.long_net_market.cumsum(), "k", lw=0.8, label="always long net")
ax[1].set_ylabel("cumulative ticks per contract"); ax[1].set_title("Always-in sign(E[r]), flat at the close"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / "01_calibration_pnl.png", dpi=110); plt.show()

In [ ]:
summary = {"registration": H.registration(CFG), "window": [str(w) for w in WINDOW], "refits": [str(r) for r in REFITS],
           "synthetic": SYNTHETIC, "synthetic_null": SYNTH_NULL, "decision": DEC,
           "primary": RES.loc["1h_H3", show].to_dict(), "secondary": RES.loc["15m_H3", show].to_dict()}
json.dump(summary, open(OUT / "01_decision.json", "w"), indent=1,
          default=lambda o: float(o) if isinstance(o, (np.floating, np.integer, np.bool_)) else str(o))
print("PRIMARY 1h hierarchy:", DEC["1h_H3"]["verdict"], "| 1h no parent:", DEC["1h_U"]["verdict"],
      "| 15m hierarchy:", DEC["15m_H3"]["verdict"], "| 15m no parent:", DEC["15m_U"]["verdict"],
      "| video rule (1h):", DEC["1h_RULE"]["verdict"])